## Section 0 — Setup & Framing

Welcome to this end-to-end tutorial on Transfer Learning with PyTorch!

**Goal:** Build a complete, pedagogically structured Jupyter notebook that teaches transfer learning by fine-tuning a pre-trained ResNet18 for binary image classification[cite: 1].
**Domain:** Manufacturing defect detection[cite: 1].
**Inputs:** A directory of RGB images (224x224 resolution) organized into `train/normal`, `train/defective`, `val/normal`, `val/defective` subfolders[cite: 1].
**Dataset Properties:** Very small — 400 training images, 100 validation images. Classes are highly imbalanced (80% normal / 20% defective)[cite: 1].
**Outputs:** 
* 4 trained model artifacts: full-freeze, layer4-unfreeze, PEFT/LoRA, and a from-scratch baseline[cite: 1].
* A final comparison table/plot (accuracy, precision/recall per class, overfitting gap) across all 4 variants[cite: 1].

> **Trade-off Note:** We prioritize memory efficiency over training speed since this notebook is designed for CPU-only execution[cite: 1]. To prevent impractically slow training times during the 4-way comparison, we strictly limit training to 2 epochs[cite: 1].

In [ ]:
import os
import glob
import random
import shutil
import urllib.request
import zipfile
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# Device detection with automatic CPU fallback[cite: 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Check for PEFT dependency, declare fallback if missing[cite: 1]
try:
    import peft
    from peft import LoraConfig, get_peft_model
    PEFT_AVAILABLE = True
    print("PEFT library found.")
except ImportError:
    PEFT_AVAILABLE = False
    print("PEFT library unavailable. Will fall back to a minimal manual adapter-injection implementation[cite: 1].")

## Section 1 — The Pre-trained Model

**What does "pre-trained" mean?**
A pre-trained model is one that has already been trained on a massive dataset to solve a general problem. Instead of starting with random weights, we download weights that already "know" how to extract useful visual features.

**Source Task & Domain:** 
ResNet18 was originally trained on ImageNet (the source task), a dataset of over 1 million natural images spanning 1,000 diverse classes (dogs, cars, keyboards, etc.). 

**Why do ImageNet features generalize?**
Deep neural networks learn hierarchically. The early layers learn to detect basic geometric primitives (edges, corners, color gradients). Because these primitives are universal to almost all visual data, a model trained on dogs and cars can seamlessly detect edges in industrial metal pipes or fabrics.

In [ ]:
# Load using the current weights API, avoiding deprecated pretrained=True[cite: 1]
model_pretrained = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

print("--- Top Level Architecture ---")
for name, child in model_pretrained.named_children():
    print(name)

print("\nTotal parameters:", sum(p.numel() for p in model_pretrained.parameters()))

## Section 2 — Source Task vs. Target Task

| Feature | Source Task | Target Task |
| :--- | :--- | :--- |
| **Dataset** | ImageNet | Manufacturing Defect Dataset[cite: 1] |
| **Domain** | Natural Images (animals, objects) | Industrial Images[cite: 1] |
| **Classes** | 1,000 Classes | 2 Classes (Binary)[cite: 1] |
| **Size** | > 1.2 Million | Very Small (400 Train / 100 Val)[cite: 1] |

### Domain Shift Checkpoint
Before touching the model, we must assess **domain shift**. Are our target images structurally similar to ImageNet? 
*   If industrial images look fundamentally different from natural images (e.g., highly specialized microscopic scans), this domain shift indicates we may need to unfreeze more layers to adapt the higher-level representations.
*   Given the small dataset size (400 training images), aggressive unfreezing risks overfitting[cite: 1]. These judgments will drive our freezing choices in Section 5.

Below, we handle dataset routing. If input folders are empty, we automatically download PyTorch’s Hymenoptera dataset (real photographic images of ants/bees) to simulate the strict 80/20 class imbalance[cite: 1].

In [ ]:
DATA_DIR = "./dataset"
subdirs = ["train/normal", "train/defective", "val/normal", "val/defective"]

for d in subdirs:
    os.makedirs(os.path.join(DATA_DIR, d), exist_ok=True)

def is_empty(dir_path):
    return len(os.listdir(dir_path)) == 0

# Automated Dataset Handling: strictly use real photographic images, DO NOT generate dummy data[cite: 1].
if all(is_empty(os.path.join(DATA_DIR, d)) for d in subdirs):
    print("Input folders empty. Downloading Hymenoptera dataset to simulate industrial dataset[cite: 1]...")
    url = 'https://download.pytorch.org/tutorial/hymenoptera_data.zip'
    urllib.request.urlretrieve(url, 'hymenoptera.zip')
    with zipfile.ZipFile('hymenoptera.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    
    # We simulate 80% normal (bees), 20% defective (ants)[cite: 1]
    bees = glob.glob('hymenoptera_data/train/bees/*.jpg') + glob.glob('hymenoptera_data/val/bees/*.jpg')
    ants = glob.glob('hymenoptera_data/train/ants/*.jpg') + glob.glob('hymenoptera_data/val/ants/*.jpg')
    
    random.seed(42)
    random.shuffle(bees)
    random.shuffle(ants)
    
    # Train: 400 total (320 normal, 80 defective)[cite: 1]
    for i, img in enumerate(bees[:320]): shutil.copy(img, os.path.join(DATA_DIR, 'train/normal', f'bee_{i}.jpg'))
    for i, img in enumerate(ants[:80]): shutil.copy(img, os.path.join(DATA_DIR, 'train/defective', f'ant_{i}.jpg'))
    
    # Val: 100 total (80 normal, 20 defective)[cite: 1]
    for i, img in enumerate(bees[320:400]): shutil.copy(img, os.path.join(DATA_DIR, 'val/normal', f'bee_{i}.jpg'))
    for i, img in enumerate(ants[80:100]): shutil.copy(img, os.path.join(DATA_DIR, 'val/defective', f'ant_{i}.jpg'))
    print("Dataset generated successfully.")

# Define augmentation transforms (Section 6 requirement mapped here for dataset loading)
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=16, shuffle=True, num_workers=0) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
print(f"Classes: {class_names}")
print(f"Dataset sizes: {dataset_sizes}")

## Section 3 — Feature Reuse: Visualizing What Transfers

To understand *why* we freeze certain layers, we can use PyTorch hooks to capture the activations as an image flows through the network.

*   **Early Layers (`layer1`):** Capture generic features like edges, curves, and textures. These are highly transferable.
*   **Late Layers (`layer4`):** Capture complex, task-specific features (e.g., dog faces vs. car wheels). These are less transferable if the target domain shifts significantly from the source.

In [ ]:
activations = {}
def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

model_pretrained.layer1[0].conv1.register_forward_hook(get_activation('early'))
model_pretrained.layer4[1].conv2.register_forward_hook(get_activation('late'))

# Pass a single target-domain image
sample_inputs, _ = next(iter(dataloaders['val']))
model_pretrained.eval()
with torch.no_grad():
    _ = model_pretrained(sample_inputs[0].unsqueeze(0))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(activations['early'][0, 0].cpu().numpy(), cmap='viridis')
axes[0].set_title("Early Layer (Generic Edges)")
axes[1].imshow(activations['late'][0, 0].cpu().numpy(), cmap='viridis')
axes[1].set_title("Late Layer (Task Specific)")
plt.show()

## Section 4 — Network Modification

In transfer learning theory, we repurpose the feature extractor (the convolutions) but replace the "classifier head" (the final Fully Connected layer). 

ResNet18 originally outputs 1,000 classes. We must slice off this final layer and attach a new `nn.Linear` layer that maps the 512 input features to our 2 target classes[cite: 1].

In [ ]:
in_features = model_pretrained.fc.in_features
model_pretrained.fc = nn.Linear(in_features, len(class_names))
print(f"Modified final layer: {model_pretrained.fc}")

## Section 5 — Freeze vs. Fine-tune vs. PEFT (Core Comparison)

Here we define our three transfer learning strategies as isolated functions. **Each function internally loads a fresh pre-trained model instance to ensure isolated weight initialization and a fair comparison[cite: 1].**

1.  **`train_full_freeze`:** Freezes all convolutional layers. Only the new head learns. This is the default strategy for a very small dataset (400 images) to prevent overfitting[cite: 1].
2.  **`train_unfreeze_layer4`:** Freezes early layers but allows `layer4` to adapt. Given the class imbalance, controlled exceptions to full freezing can sometimes help the network discover distinct anomaly features, but require careful tuning[cite: 1].
3.  **`train_peft`:** Uses LoRA (Low-Rank Adaptation) via the `peft` library. If missing, applies a minimal manual adapter (bottleneck) on the FC layer[cite: 1].

In [ ]:
def train_full_freeze(dataloaders, criterion, device, num_epochs=2):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 2) # Section 4 Modification
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
    return train_model(model, dataloaders, criterion, optimizer, device, num_epochs, 'full_freeze')

def train_unfreeze_layer4(dataloaders, criterion, device, num_epochs=2):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for name, param in model.named_parameters():
        if 'layer4' not in name and 'fc' not in name:
            param.requires_grad = False
            
    model.fc = nn.Linear(model.fc.in_features, 2)
    model = model.to(device)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)
    return train_model(model, dataloaders, criterion, optimizer, device, num_epochs, 'layer4_unfreeze')

def train_peft(dataloaders, criterion, device, num_epochs=2):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 2)
    
    if PEFT_AVAILABLE:
        config = LoraConfig(
            r=8, lora_alpha=16, target_modules=["conv1", "conv2"], 
            lora_dropout=0.1, bias="none", modules_to_save=["fc"]
        )
        try:
            model = get_peft_model(model, config)
        except Exception as e:
            print(f"PEFT injection failed: {e}. Falling back to manual adapter[cite: 1].")
            global PEFT_AVAILABLE
            PEFT_AVAILABLE = False
            
    if not PEFT_AVAILABLE:
        for param in model.parameters(): 
            param.requires_grad = False
        orig_fc = model.fc
        model.fc = nn.Sequential(
            nn.Linear(orig_fc.in_features, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )
        
    model = model.to(device)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
    return train_model(model, dataloaders, criterion, optimizer, device, num_epochs, 'peft_adapter')

## Section 6 — Training Loop with Class Imbalance Handling

Because our classes are heavily skewed (80% / 20%), a naive model will optimize by simply predicting the majority class every time. 

We counter this using **Weighted CrossEntropyLoss**. We assign a proportionally higher weight to the minority `defective` class[cite: 1].
$\text{Loss} = -w_c \log(p_c)$

At the bottom of this section, an orchestrating cell runs our 3 variants defined above[cite: 1].

In [ ]:
# Imbalance Handling (80% normal vs 20% defective) -> Inverse weights
# Assuming class 0 = defective, 1 = normal based on alphabetical sorting (ants=defective, bees=normal)
# Actually, lets calculate strictly based on dataset presence
class_counts = [len(glob.glob(f"{DATA_DIR}/train/{c}/*.jpg")) for c in class_names]
total = sum(class_counts)
class_weights = torch.FloatTensor([total / c for c in class_counts]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

def train_model(model, dataloaders, criterion, optimizer, device, num_epochs, variant_name):
    print(f"\n--- Training Variant: {variant_name} ---")
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()
                
            running_loss = 0.0
            running_corrects = 0
            
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                        
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {history['train_loss'][-1]:.4f} Acc: {history['train_acc'][-1]:.4f} | Val Loss: {history['val_loss'][-1]:.4f} Acc: {history['val_acc'][-1]:.4f}")
        
    torch.save(model.state_dict(), f"{variant_name}.pth")
    return model, history

In [ ]:
# Orchestrating cell sequentially calling Section 5 functions[cite: 1]
EPOCHS = 2 # Forced low specifically for 4-way CPU comparison efficiency[cite: 1]

results = {}
results['full_freeze'] = train_full_freeze(dataloaders, criterion, device, num_epochs=EPOCHS)
results['layer4_unfreeze'] = train_unfreeze_layer4(dataloaders, criterion, device, num_epochs=EPOCHS)
results['peft_adapter'] = train_peft(dataloaders, criterion, device, num_epochs=EPOCHS)

## Section 7 — Baseline, Evaluation & QA

We train a completely uninitialized model from scratch as a control baseline[cite: 1]. Finally, we extract and plot Precision/Recall and Overfitting metrics.

In [ ]:
def train_baseline(dataloaders, criterion, device, num_epochs=2):
    model = models.resnet18(weights=None) # From scratch, uninitialized
    model.fc = nn.Linear(model.fc.in_features, 2)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    return train_model(model, dataloaders, criterion, optimizer, device, num_epochs, 'baseline')

results['baseline'] = train_baseline(dataloaders, criterion, device, num_epochs=EPOCHS)

def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(inputs.to(device))
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Handle zero division gracefully[cite: 1]
    precision, recall, _, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, zero_division=0)
    return precision, recall

print("\n--- Final Metrics ---")
for variant, (model, hist) in results.items():
    p, r = evaluate_model(model, dataloaders['val'], device)
    overfit_gap = hist['train_acc'][-1] - hist['val_acc'][-1]
    print(f"{variant}: Prec={np.round(p, 2)}, Rec={np.round(r, 2)} | Overfit Gap: {overfit_gap:.4f}")

### Sanity Check: Did transfer learning help?
Did any transfer variant (freeze, unfreeze, PEFT) outperform the from-scratch baseline?[cite: 1]
If not (or if results are tied), this is often because 2 epochs is insufficient for the models to converge, or because the from-scratch model collapsed into predicting only the majority class. Examine your Precision/Recall per class—baseline likely has a Recall of 0.0 for the minority class, proving that transfer learning provides a vastly superior inductive bias.

---
## Section 8 — Reflection: When Would Transfer Fail?

Revisiting our relatedness assumption from Section 2, consider a target task entirely alien to ImageNet—such as RF spectrogram classification or volumetric Medical MRI scans[cite: 1]. 

In those domains, standard transfer learning often leads to **Negative Transfer**. The generic "edges" learned from dogs and cars actively interfere with finding nuanced RF signal noise. In those cases, a full-freeze strategy would fail spectacularly, and you would be better off utilizing an architecture pre-trained on in-domain data or training from scratch.